In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("data/powerplant_data.csv")

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
#AT - temperature
#V - vaccum
# AP - pressure
#RH - humidity

In [5]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [6]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state = 42
)

In [8]:
df.shape

(9568, 5)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## DeepLearning

In [13]:
# Build our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

In [14]:
import torch.optim as optim

model = ANN()

crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):

    # --------------------
    # Training
    # --------------------
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:

        # Forward propagation
        outputs = model(xb)

        # Calculate loss
        loss = crietrion(outputs, yb)

        # Backpropagation
        optimizer.zero_grad()   # IMPORTANT
        loss.backward()

        # Update parameters
        optimizer.step()

        # Add batch loss
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # --------------------
    # Validation
    # --------------------
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        for xb, yb in test_loader:

            outputs = model(xb)

            loss = crietrion(outputs, yb)

            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"==> Training Loss = {epoch_train_loss:.4f} "
        f"| Val Loss = {epoch_val_loss:.4f}"
    )

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "models/best_model.pt") #.pt or .ph

Epoch 1/100 ==> Training Loss = 206317.7471 | Val Loss = 204650.5383
Epoch 2/100 ==> Training Loss = 197782.2683 | Val Loss = 185471.9151
Epoch 3/100 ==> Training Loss = 162464.7563 | Val Loss = 134448.1245
Epoch 4/100 ==> Training Loss = 100943.7386 | Val Loss = 69778.4464
Epoch 5/100 ==> Training Loss = 47832.2353 | Val Loss = 32800.2995
Epoch 6/100 ==> Training Loss = 26400.1972 | Val Loss = 22508.5943
Epoch 7/100 ==> Training Loss = 20496.1375 | Val Loss = 18489.4650
Epoch 8/100 ==> Training Loss = 16867.4687 | Val Loss = 15035.3832


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd # Added missing pandas import

# Ensure the keys here...
loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss": val_losses
})

# ...match the keys you call here
plt.plot(loss_df["Training Loss"], label="Training Loss")
plt.plot(loss_df["Validation Loss"], label="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()
plt.show() # Added to actually display the plot

In [ ]:
model.load_state_dict(torch.load("models/best_model.pt"))

In [ ]:
#Evaluation

model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = crietrion(train_preds, y_train_tensor)
    test_mse_loss = crietrion(test_preds, y_test_tensor)

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("r^2 score=", r2_score(y_test, test_preds))

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

In [ ]:
pd.concat([predicted_df, actual_df], axis=1)